In [8]:
import json
import re
import pathlib

import numpy as np
import pandas as pd

def read_log_file(path: str | pathlib.Path) -> tuple[str, str]:
    path = pathlib.Path(path)
    raw = path.read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("ISO-8859-1"), "ISO-8859-1"

LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$"
)


def parse_log(text: str) -> list[dict]:
    """Extract valid transaction records from log text, ignoring noise lines."""
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

# Task 2

def find_log_files(directory: str | pathlib.Path, extension: str = ".log") -> list[pathlib.Path]:
    """Recursively find all files with the given extension (no glob/rglob/os.walk)."""
    directory = pathlib.Path(directory)
    found = []
    for item in directory.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))   # recursive call
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    return found


def find_active_registers(node) -> set[str]:
    """Recursively traverse nested dicts/lists and collect ids where active is True."""
    active: set[str] = set()
    if isinstance(node, dict):
        if node.get("active") is True and "id" in node:
            active.add(node["id"])
        for value in node.values():
            active |= find_active_registers(value)
    elif isinstance(node, list):
        for item in node:
            active |= find_active_registers(item)
    return active

# Task 3

def remove_price_outliers(prices) -> np.ndarray:
    """Replace values with |x - mean| > 3*std by the median of the valid values."""
    arr = np.asarray(prices, dtype=float).ravel()     # 1D float array
    mu, sigma = arr.mean(), arr.std()
    is_outlier = np.abs(arr - mu) > 3 * sigma         # Boolean mask
    cleaned = arr.copy()
    if is_outlier.any():
        cleaned[is_outlier] = np.median(arr[~is_outlier])
    return cleaned


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Element-wise refund ratio with epsilon to avoid division by zero."""
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)

# Task 4

def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """List of dicts -> DataFrame with datetime index, sorted chronologically."""
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.set_index("timestamp").sort_index()


def hourly_revenue_per_item(df: pd.DataFrame) -> pd.Series:
    """Total price per item in 1-hour windows."""
    return df.groupby("item")["price"].resample("1h").sum()


def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item (rows) across status categories (columns)."""
    return pd.pivot_table(df, values="price", index="item", columns="status",
                          aggfunc="sum", fill_value=0)


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item = #REFUND / #SUCCESS, via compute_refund_ratio."""
    counts = df.groupby(["item", "status"]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)
    ratios = compute_refund_ratio(counts["SUCCESS"].to_numpy(),
                                  counts["REFUND"].to_numpy())
    return pd.DataFrame({
        "sales_count": counts["SUCCESS"],
        "refund_count": counts["REFUND"],
        "refund_ratio": ratios.round(4),
    })

# Task 5

def run_pipeline(data_dir="data", output_dir="output"):
    """Full workflow: find -> parse -> DataFrame -> clean -> correct -> analyse -> save."""
    data_dir = pathlib.Path(data_dir)
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Find log files and active registers
    log_files = find_log_files(data_dir)
    with open(data_dir / "store_config.json", encoding="utf-8") as f:
        active_registers = find_active_registers(json.load(f))

    # 2. Read and parse all files
    records = []
    for path in log_files:
        text, encoding = read_log_file(path)
        records.extend(parse_log(text))

    # 3. Build the DataFrame
    df = build_dataframe(records)

    # 4. Remove 999 outliers (BEFORE currency correction)
    df["price"] = remove_price_outliers(df["price"].to_numpy())

    # 5. Currency correction: NOK values are recorded on the EUR scale -> multiply by 10
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    # 6. Generate reports
    hourly = hourly_revenue_per_item(df)
    pivot = status_pivot_table(df)
    ratios = item_refund_ratios(df)

    # 7. Save files
    df.round(2).to_csv(output_dir / "cleaned_transactions.csv")

    wide = hourly.unstack(level="item", fill_value=0)   # rows: time, columns: item
    hourly_report = {
        item: {ts.strftime("%Y-%m-%d %H:%M"): round(float(v), 2)
               for ts, v in wide[item].items()}
        for item in wide.columns
    }
    with open(output_dir / "hourly_revenue.json", "w", encoding="utf-8") as f:
        json.dump(hourly_report, f, indent=2)

    pivot.round(2).to_csv(output_dir / "status_pivot_table.csv")
    ratios.to_csv(output_dir / "item_refund_ratios.csv")

    return df, hourly, pivot, ratios, active_registers



if __name__ == "__main__":
    df, hourly, pivot, ratios, active = run_pipeline()
    print("Active registers:", sorted(active))
    print(df.groupby("currency")["price"].describe().round(2))
    print(pivot.round(2))
    print(ratios)


Active registers: ['POS_01', 'POS_02', 'POS_03']
          count   mean    std    min    25%   50%    75%    max
currency                                                       
EUR       239.0   4.45   1.54   2.51   3.42   4.0   4.70   7.91
NOK       250.0  44.02  15.21  25.10  34.02  40.0  46.10  79.80
USD       231.0   4.36   1.49   2.53   3.38   4.0   4.59   7.96
status      CANCELLED  REFUND  SUCCESS
item                                  
Cappuccino     306.96  183.14  1298.13
Croissant      406.07  291.74  1032.88
Espresso       255.56  338.49  1275.23
Latte          242.09  418.11  1570.36
Muffin         209.59  375.30  1020.18
Sandwich       506.24  627.43  2719.91
            sales_count  refund_count  refund_ratio
item                                               
Cappuccino           73            17        0.2329
Croissant            63            21        0.3333
Espresso             88            27        0.3068
Latte                89            19        0.2135
Muffin 